# A2.4 · Just-in-time authority

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.3 · Delegation that narrows, and survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**.

| | |
|---|---|
| Tools used | Keycloak, OPA |

## What this lesson is

**What it covers.** Issue a scoped grant, use it, then replay it after expiry and after the task closed.

**Why a security engineer needs it.** Permanent scope makes every injection a successful one, because the authority is always there when the attacker arrives. The control it builds is: short-lived, purpose-bound grants issued per task and expiring with it.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Standing authority means a successful injection always finds a live credential waiting. Just-in-time authority means the attacker has to arrive during the ninety seconds the grant exists, and be doing the one task it was scoped to.

> **At CyberTravels.** Standing payments scope means a successful injection always finds a live refund credential. Just-in-time means the attacker has to arrive during the ninety seconds a specific booking is being settled. R1, R5.

## 2 · The framework

```
   standing                       just-in-time
   +-----------------+            +---------------------+
   | granted once    |            | granted per task    |
   | lives forever   |            | expires in 90s      |
   | any task        |            | this resource only  |
   +-----------------+            +---------------------+
   injection always finds         injection has to arrive
   a live credential              during the window, on that task
```

**Mitigates: T3 Privilege Compromise · T2 Tool Misuse.**

A2.3 narrows authority at the moment of delegation. This lesson removes it when
the task is over.

Standing authority is the reason an injection is always worth attempting: the
credential is there, permanently, waiting. Every successful A1.3 lands on a live
grant. Just-in-time authority changes the arithmetic — the attacker has to
arrive during a window that exists only while a specific task is running, and
that is bound to a specific resource.

Three properties, and the third is the one usually skipped:

**Short-lived.** Minutes, not months. Theft has a deadline.

**Purpose-bound.** Scoped to *this* resource, not to the resource class. Not
`reports:write` but `reports:write` on report 8812.

**Revoked on completion.** The grant ends when the task ends, not when the timer
does. A task that finishes in ten seconds should not leave a fifteen-minute
credential lying around, which is the difference between a TTL and an actual
lifecycle.

The operational cost is real and worth stating plainly: something must issue
these, and if that path breaks, work stops. That is the trade — a system that
fails closed under a control outage, in exchange for a system that has no
standing authority to steal.

> **What this control closes.**
>
> Removes the **standing** grant an injection needs. The attacker must now arrive inside a window bound to one task and one resource.

## 3 · Finding the standing grants, as a skill

Just-in-time authority is only worth building where standing authority exists today, and at CyberTravels that list is not the one in the design document — it is in the authorisation graph and in the OAuth scopes the credential provider stored when someone first connected the payments API. The procedure diffs what each identity *holds* against what its declared tools actually *need*, and flags every permanent grant. This is the file in this repository:

### The skill — [`skills/attestation/entitlement-overprivilege-analyzer/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/entitlement-overprivilege-analyzer/SKILL.md)

```yaml
name: entitlement-overprivilege-analyzer
description: >-
  Analyse entitlement and identity registries to find over-privileged access
  held by a deployment's identities. Use to check granted entitlements
  against what the declared tools actually need, to find over-broad OAuth
  scopes on credential providers, or to find standing privilege.
allowed-tools: Bash, Read
```

# Entitlement Overprivilege Analyzer

**Controls:** Controls 1 and 3 — over-privileged access

## What this adds beyond the IAM verifier

The IAM verifier measures cloud permissions against cloud usage. This skill
measures **application-level entitlements** — relationship-graph authorisation,
OAuth scopes on stored credential providers, and identity-registry
relationships — against what the declared tool surface actually requires.

An agent can hold a minimal cloud role and an OAuth token with full mailbox
access.

## When to use this
After the IAM baseline is verified, not instead of it. The IAM verifier asks
whether the role is default-deny; this asks whether the entitlements actually
granted exceed what the declared tools need. Reach for it when scopes were
granted at integration time, when a credential provider holds OAuth scopes
nobody has reviewed, or when looking for standing privilege.

## Procedure

1. **Take the required capability set** from the code-surface analyzer. This is
   the denominator: what the declared tools genuinely need.
2. **Enumerate granted entitlements** from the authorisation graph for every
   identity in the deployment manifest.
3. **Enumerate credential-provider scopes.** Stored OAuth scopes are frequently
   far wider than the tool needs, because the consent screen offered a bundle.
4. **Diff.** Every grant with no corresponding requirement is an over-privilege
   finding, and each needs a justification gap recorded — the grant, the
   requirement it was presumably for, and the absence.
5. **Flag standing privilege.** Any grant that is permanent rather than issued
   per task.

## Example

**Input** — the fixture committed at the top of [`scripts/entitlement_overprivilege_analyzer.py`](scripts/entitlement_overprivilege_analyzer.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
   the task's own write      ok      permitted
   a different report        REFUSED bound to report/8812
   a different scope         REFUSED scoped to reports:write
   after the task completes  REFUSED task closed
   after the TTL expires     REFUSED expired

An injection landing at 09:14 needs a task to be open, on the resource
it wants, holding the scope it wants. Standing authority required none
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "deployment_id": "str",
  "identities": [
    {"id": "str",
     "granted": ["str"],
     "required_by_tools": ["str"],
     "excess": [{"grant": "str", "justification_gap": "str"}]}
  ],
  "oauth_scope_excess": [{"provider": "str", "granted_scope": "str", "needed_scope": "str"}],
  "standing_privilege": ["str"],
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Comparing grants against other grants.** The denominator is the tool
  surface, not a peer deployment.
- **Accepting a bundled OAuth scope** because it was what the provider offered.
- **Missing standing privilege** because the scope itself looked narrow.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/entitlement-overprivilege-analyzer/scripts/entitlement_overprivilege_analyzer.py
SCRIPT = "skills/attestation/entitlement-overprivilege-analyzer/scripts/entitlement_overprivilege_analyzer.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The skill loads and reports its shape. Note what its procedure insists on: the denominator is the capability set the tools require, not another set of grants — comparing grants against grants is how a review concludes that an over-privileged agent is normal — and a narrow-looking scope still counts as standing privilege if it never expires.

## Your turn

Take one standing grant an agent holds and work out what would break if it expired in two minutes. That list is the real cost of just-in-time, and it is usually shorter than expected.

---

**Next → [A2.5 · The non-human identity lifecycle](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*